## Load the tokenizer

In [ ]:
import sys
import os
sys.path.append('..')

In [ ]:
from minbpe.minbpe import BasicTokenizer

tokenizer = BasicTokenizer()
tokenizer.load(model_file="./output/tokenizer/my_tokenizer.model")


def get_vocab_size(tokenizer: BasicTokenizer) -> int:
    vocab = tokenizer.vocab
    special_tokens = tokenizer.special_tokens

    return len(vocab) + len(special_tokens)

## Create the model

In [ ]:
import torch
torch.manual_seed(3647)

In [ ]:
import torch.version


torch.version.__version__

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Config
import torch

# Hyperparameters
block_size = 256
n_embd = 128
n_head = 8    
n_layer = 4
dropout = 0.2
batch_size = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'
vocab_size=get_vocab_size(tokenizer)

# Configure GPT-2 model
config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=block_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    dropout=dropout,
    bos_token_id=tokenizer.encode("<|BOS|>")[0] if "<|BOS|>" in tokenizer.special_tokens else None,
    eos_token_id=tokenizer.encode("<|EOS|>")[0] if "<|EOS|>" in tokenizer.special_tokens else None
)

# Initialize prebuilt GPT-2 model
model = GPT2LMHeadModel(config=config).to(device)

# Compile model (requires PyTorch 2.0+)
# try:
#     model = torch.compile(model)
# except Exception as e:
#     print(f"torch.compile failed: {e}. Proceeding without compilation.")

# Calculate and print number of parameters
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"{total_params:.2f} M parameters")

## Data preparation

### 1. Load the data

In [6]:
with open("./output/combined_text.txt", "r",encoding="utf-8") as f:
    text_sequence = f.read()

encoded_text_sequence = tokenizer.encode(text_sequence)
len(encoded_text_sequence)

321451

### 2. Split it into train and test

In [7]:
data = torch.tensor(encoded_text_sequence, dtype=torch.long)
split_index = int(0.9*len(data))
train_data = data[:split_index]
val_data = data[split_index:]

### 3. Data loader

In [8]:
from typing import Tuple
from torch.utils.data import Dataset, DataLoader


class TextDataset(Dataset):
    def __init__(self, data: torch.Tensor, block_size: int) -> None:
        self.data = data
        self.block_size = block_size

    def __len__(self) -> int:
        return len(self.data) - self.block_size

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.data[index:index + self.block_size]
        y = self.data[index + 1:index + self.block_size + 1]
        return x, y


def get_dataloaders(
        train_data: torch.Tensor,
        val_data: torch.Tensor,
        block_size: int,
        batch_size: int,
        device: torch.device
) -> Tuple[DataLoader, DataLoader]:
    train_dataset = TextDataset(train_data.to(device), block_size)
    val_dataset = TextDataset(val_data.to(device), block_size)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    return train_loader, val_loader

In [9]:
train_loader, val_loader = get_dataloaders(
    train_data=train_data,
    val_data=val_data,
    block_size=block_size,
    batch_size=batch_size,
    device=device
)
x, y = next(iter(train_loader))
x.shape, y.shape

print(f"Train loader batches: {len(train_loader)}")
print(f"Val loader batches: {len(val_loader)}")
if len(train_loader) == 0 or len(val_loader) == 0:
    raise ValueError("Empty DataLoader. Increase text length or reduce block_size.")

Train loader batches: 4517
Val loader batches: 499


In [10]:
# Test one batch
try:
    x_batch, y_batch = next(iter(train_loader))
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
    outputs = model(input_ids=x_batch, labels=y_batch)
    print(f"Test batch loss: {outputs.loss.item()}")
except Exception as e:
    print(f"Error testing batch: {e}")
    raise

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Test batch loss: 6.962081432342529


### 4. Training

In [11]:
from typing import Dict


# Updated estimate_loss (simplified for GPT2LMHeadModel)
@torch.no_grad()
def estimate_loss(
    model: torch.nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    eval_iters: int,
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
) -> Dict[str, float]:
    output = {}
    model.eval()
    for split, loader in [('train', train_loader), ('val', val_loader)]:
        losses = []
        for i, (x, y) in enumerate(loader):
            if i >= eval_iters:
                break
            x, y = x.to(device), y.to(device)
            outputs = model(input_ids=x, labels=y)
            losses.append(outputs.loss.item())
        output[split] = sum(losses) / len(losses) if losses else float('inf')
    model.train()
    return output

In [12]:
# Updated save_checkpoint
def save_checkpoint(
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    loss: float,
    file_path: str = "./checkpoints/checkpoint.pth"
) -> None:
    os.makedirs(os.path.dirname(file_path) or ".", exist_ok=True)
    model_state_dict = model.state_dict()  # No _orig_mod needed without torch.compile
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model_state_dict,
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss
    }
    torch.save(checkpoint, file_path)
    print(f"Checkpoint saved to {file_path}")

In [ ]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True

In [ ]:
import json 

# Training hyperparameters
max_iters = 1
eval_interval = 100
eval_iters = 200
learning_rate = 3e-4

# Initialize optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training loop
train_losses = []
val_losses = []

for iteration in range(max_iters):
    print("Iteration")
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        
        # Evaluation
        if batch_idx % eval_interval == 0 or batch_idx == len(train_loader) - 1:
            losses = estimate_loss(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                eval_iters=min(eval_iters, len(val_loader)),
                device=device
            )
            train_losses.append(losses['train'])
            val_losses.append(losses['val'])
            print(
                f"iteration {iteration} / step {batch_idx}: "
                f"train loss {losses['train']:.4f}, "
                f"val loss {losses['val']:.4f}"
            )
        
        # Training step
        outputs = model(input_ids=x_batch, labels=y_batch)
        loss = outputs.loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    
    # Save checkpoint
    checkpoint_path = f"./output/pre_training/run_4/checkpoint_{iteration}.pth"
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        epoch=iteration,
        loss=loss.item(),
        file_path=checkpoint_path
    )

# Save loss history
with open("./output/pre_training/run_4/losses.json", "w") as f:
    json.dump({"train_losses": train_losses, "val_losses": val_losses}, f)
print("Loss history saved.")

Iteration
iteration 0 / step 0: train loss 6.9618, val loss 6.9634
iteration 0 / step 100: train loss 6.2222, val loss 6.2985
iteration 0 / step 200: train loss 6.0184, val loss 6.0433
iteration 0 / step 300: train loss 5.8659, val loss 5.8950
iteration 0 / step 400: train loss 5.7538, val loss 5.7932
iteration 0 / step 500: train loss 5.6668, val loss 5.7360
iteration 0 / step 600: train loss 5.6011, val loss 5.6938
iteration 0 / step 700: train loss 5.5342, val loss 5.6630
iteration 0 / step 800: train loss 5.4682, val loss 5.6283
iteration 0 / step 900: train loss 5.4049, val loss 5.6106
iteration 0 / step 1000: train loss 5.3480, val loss 5.5948
iteration 0 / step 1100: train loss 5.2849, val loss 5.5972
iteration 0 / step 1200: train loss 5.2325, val loss 5.5939
iteration 0 / step 1300: train loss 5.1757, val loss 5.5710


In [ ]:
max_iters = 1
eval_interval = 100
eval_iters = 200
learning_rate = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
train_loader, val_loader = get_dataloaders(
    train_data=train_data,
    val_data=val_data,
    block_size=block_size,
    batch_size=batch_size,
    device=device
)

train_losses = []
val_losses = []

for iteration in range(max_iters):
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        # Evaluation
        if batch_idx % eval_interval == 0 or batch_idx == len(train_loader) - 1:
            losses = estimate_loss(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
                eval_iters=min(eval_iters, len(val_loader))
            )
            train_losses.append(losses['train'])
            val_losses.append(losses['val'])

            print(
                f"iteration {iteration} / step {batch_idx}: "
                f"train loss {losses['train']:.4f}, "
                f"val loss {losses['val']:.4f}"
            )

        # Training step
        logits, loss = model(x_batch, y_batch)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # Save checkpoint
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        epoch=iteration,
        loss=loss.item(),
        file_path=f"./output/pre_training/run_4/checkpoint_{iteration}.pth"
    )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss", marker='o')
plt.plot(val_losses, label="Validation Loss", marker='o')
plt.xlabel("Evaluation Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Over Time")
plt.legend()
plt.grid()
plt.show()

In [ ]:
input_tokens = tokenizer.encode("Salam labas ")
input_tokens = torch.tensor(
    input_tokens, dtype=torch.long).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=100)

print(tokenizer.decode(output[0].tolist()))